# Comprehensive Results Summary

Aggregates every training run and test evaluation into a single indexed table,
keyed by run name.

**Reads**
- `results/*_history.npy` — per-fold training histories (validation metrics)
- `results/test_eval_*.npz` — per-fold test confusion matrices
- `results/latency_pi.json` — edge-device latency (optional)

**Writes**
- `results/summary_per_fold.md` / `.csv`
- `results/summary_per_config.md` / `.csv`

Run top to bottom. Configurations that were planned but never trained are
reported in the coverage section rather than silently omitted.

## 1 · Configuration

In [121]:
# !pip install prettytable  # uncomment if not installed

import json
import re
import warnings
from collections import OrderedDict
from pathlib import Path

import numpy as np
from prettytable import PrettyTable

RESULTS_DIR  = Path("../results")     # adjust if running from repo root
LATENCY_JSON = RESULTS_DIR / "latency_pi.json"
OUT_DIR      = RESULTS_DIR

STRATEGY   = "vpfabelo"
FOLDS      = [1, 2, 3, 4, 5]

# Confusion-matrix label order and semantics
CM_LABELS  = [1, 2, 3, 4]              # NT, TT, BV, BG
CLASS_ABBR = {1: "NT", 2: "TT", 3: "BV", 4: "BG"}
BG_INDEX   = 3                         # position of BG within CM_LABELS
TT_INDEX   = 1                         # position of TT within CM_LABELS

PCT = 100.0                            # report metrics as percentages

## 2 · Model registry and run-name parsing

Run names follow `{model}_{loss}_{balance}_fold{n}_{strategy}`. The model
prefix is matched against the registry below; edit it if a prefix in your
`results/` directory is missing or a parameter count is wrong.

Parameter counts marked `None` are unknown (typically because the model was
never trained) and render as `--`.

In [122]:
# prefix → (display name, tier, params, input type)
MODEL_REGISTRY = OrderedDict([
    # Tier 1 — spectral fully connected
    ("1dnnfabelo",       ("1D-NN-Fabelo",      1,      4_936, "pixel")),
    ("1dnnbaseline",     ("1D-NN-Baseline",    1, 17_000_000, "pixel")),
    ("1dnn",             ("1D-NN-Baseline",    1, 17_000_000, "pixel")),
    # Tier 2 — spectral convolutional
    ("1dcnnhu",          ("1D-CNN-Hu",         2,     76_824, "pixel")),
    ("1dcnn",            ("1D-CNN-Hu",         2,     76_824, "pixel")),
    # Tier 3 — spatial 2D
    ("2dcnnfabelo",      ("2D-CNN-Fabelo",     3,    142_052, "patch")),
    ("2dcnnsimple",      ("2D-CNN-Simple",     3,     19_644, "patch")),
    ("2dcnnlee",         ("2D-CNN-LeeEtAl",    3,    296_580, "patch")),
    ("2dcnnleeetal",     ("2D-CNN-LeeEtAl",    3,    296_580, "patch")),
    ("2dcnn",            ("2D-CNN-LeeEtAl",    3,    296_580, "patch")),
    # Tier 4 — spatio-spectral 3D
    ("3dcnn",            ("3D-CNN-Hamida",     4,       None, "patch")),
    ("hamida3dcnn",      ("3D-CNN-Hamida",     4,       None, "patch")),
    ("hybridsn",         ("HybridSN",          4,       None, "patch")),
    # Tier 5 — transformer
    ("spectralformervit",("SpectralFormer-ViT",5,       None, "patch")),
    ("sfvit",            ("SpectralFormer-ViT",5,       None, "patch")),
    ("spectralformercaf",("SpectralFormer-CAF",5,       None, "patch")),
    ("sfcaf",            ("SpectralFormer-CAF",5,       None, "patch")),
    ("spectralformer",   ("SpectralFormer",    5,       None, "patch")),
])

LOSS_NAMES = {"ce": "CE", "fl": "FL", "dl": "DL", "ufl": "UFL"}

# Match on underscore-stripped names so "spectralformer_vit" resolves to the
# ViT entry rather than falling through to the generic "spectralformer".
# Longest first so "1dcnnhu" wins over "1dcnn".
_NORM = lambda s: s.replace("_", "").replace("-", "").lower()
_SORTED_PREFIXES = sorted(MODEL_REGISTRY, key=len, reverse=True)

RUN_RE = re.compile(
    r"^(?P<model>.+?)_(?P<loss>ce|fl|dl|ufl)_(?P<bal>bal|nobal)"
    r"_fold(?P<fold>\d+)_(?P<strategy>.+)$"
)

UNKNOWN_PREFIXES = set()
NAME_MISMATCH = []


def lookup_model(prefix):
    """Resolve a model prefix to (name, tier, params, type)."""
    np_ = _NORM(prefix)
    for key in _SORTED_PREFIXES:
        nk = _NORM(key)
        if np_ == nk or np_.startswith(nk):
            return MODEL_REGISTRY[key]
    UNKNOWN_PREFIXES.add(prefix)
    return (prefix, 0, None, "unknown")


def parse_run(run_name):
    """Split a run name into its components. Returns None if unparseable."""
    m = RUN_RE.match(run_name)
    if not m:
        return None
    name, tier, params, mtype = lookup_model(m.group("model"))
    return {
        "run":        run_name,
        "config":     f"{m.group('model')}_{m.group('loss')}_{m.group('bal')}_{m.group('strategy')}",
        "prefix":     m.group("model"),
        "model":      name,
        "tier":       tier,
        "params":     params,
        "type":       mtype,
        "loss":       LOSS_NAMES.get(m.group("loss"), m.group("loss").upper()),
        "balance":    m.group("bal"),
        "fold":       int(m.group("fold")),
        "strategy":   m.group("strategy"),
    }


# Quick self-check
for _n in ["1dnnfabelo_ce_bal_fold3_vpfabelo",
           "2dcnnfabelo_ufl_bal_fold1_vpfabelo",
           "spectralformer_vit_ce_bal_fold2_vpfabelo"]:
    p = parse_run(_n)
    print(f"{_n:45} -> {p['model'] if p else 'UNPARSEABLE'}, "
          f"{p['loss'] if p else ''}, fold {p['fold'] if p else ''}")

1dnnfabelo_ce_bal_fold3_vpfabelo              -> 1D-NN-Fabelo, CE, fold 3
2dcnnfabelo_ufl_bal_fold1_vpfabelo            -> 2D-CNN-Fabelo, UFL, fold 1
spectralformer_vit_ce_bal_fold2_vpfabelo      -> SpectralFormer-ViT, CE, fold 2


## 3 · Discovery

Scans `results/` and reports what was found before any parsing happens, so a
missing or misnamed file surfaces here rather than as a confusing gap later.

In [123]:
history_files   = sorted(RESULTS_DIR.glob("*_history.npy"))
testeval_files  = sorted(RESULTS_DIR.glob("test_eval_*.npz"))
testmetric_files = sorted(RESULTS_DIR.glob("*_test_metrics.npy"))

print(f"Results directory : {RESULTS_DIR.resolve()}")
print(f"History files     : {len(history_files)}")
print(f"Test eval files   : {len(testeval_files)}  (test_eval_*.npz)")
print(f"Test metric files : {len(testmetric_files)}  (*_test_metrics.npy)")
print(f"Latency JSON      : {'found' if LATENCY_JSON.exists() else 'MISSING (optional)'}")

if not history_files:
    raise FileNotFoundError(
        f"No *_history.npy found in {RESULTS_DIR.resolve()}. "
        "Adjust RESULTS_DIR in the configuration cell."
    )

# Inspect a per-run test-metrics file so its layout is visible
if testmetric_files:
    _tm = np.load(testmetric_files[0], allow_pickle=True)
    _tm = _tm.item() if _tm.shape == () else _tm
    print()
    print('Layout of ' + testmetric_files[0].name + ':')
    if isinstance(_tm, dict):
        for k, v in sorted(_tm.items()):
            print(f'  {k:20} {type(v).__name__:10} shape={np.shape(v)}')
    else:
        print(f'  (array) shape={np.shape(_tm)} dtype={_tm.dtype}')

# Inspect one test-eval archive so its layout is visible
if testeval_files:
    _probe = np.load(testeval_files[0], allow_pickle=True)
    print(f"\nLayout of {testeval_files[0].name}:")
    for k in _probe.files:
        print(f"  {k:20} shape={np.shape(_probe[k])}  dtype={_probe[k].dtype}")

Results directory : /Users/joshua/Repositories/brain-vision/results
History files     : 111
Test eval files   : 22  (test_eval_*.npz)
Test metric files : 45  (*_test_metrics.npy)
Latency JSON      : MISSING (optional)

Layout of 1dcnn_ce_bal_fold1_vpfabelo_test_metrics.npy:
  confusion_matrix     ndarray    shape=(4, 4)
  dice                 ndarray    shape=(4,)
  macro_dice           float      shape=()
  macro_dice_no_bg     float      shape=()
  macro_f1             float      shape=()
  macro_f1_no_bg       float      shape=()
  oa                   float      shape=()
  run_name             str        shape=()
  sensitivity          ndarray    shape=(4,)
  specificity          ndarray    shape=(4,)

Layout of test_eval_1dcnn_ce_bal_vpfabelo.npz:
  run_stem             shape=()  dtype=<U21
  model                shape=()  dtype=<U6
  loss                 shape=()  dtype=<U2
  strategy             shape=()  dtype=<U9
  kind                 shape=()  dtype=<U5
  folds              

## 4 · Validation metrics from training histories

`best_f1` is macro F1 excluding background; `best_f1_all` includes it. Both
are taken at the best epoch, which is also recorded.

In [124]:
def load_history(path):
    """Extract per-fold validation metrics from a training history."""
    h = np.load(path, allow_pickle=True).item()

    # The FILENAME is authoritative. Some runs (e.g. SpectralFormer ViT vs CAF)
    # share an identical stored `run_name`, so keying on it silently collapses
    # distinct configurations into one row.
    run_name   = path.stem.replace("_history", "")
    stored     = str(h.get("run_name", run_name))
    if stored != run_name:
        NAME_MISMATCH.append((run_name, stored))

    parsed = parse_run(run_name)
    if parsed is None:
        warnings.warn(f"Could not parse run name: {run_name}")
        return None
    parsed["stored_run_name"] = stored

    n_epochs  = len(h.get("val_f1_no_bg", []))
    best_ep   = int(h.get("best_epoch", 0))
    train_loss = h.get("train_loss", [])

    parsed.update({
        "val_f1_nobg":  float(h["best_f1"]) * PCT,
        "val_f1_all":   float(h["best_f1_all"]) * PCT,
        "val_sens":     float(h["best_sens"]) * PCT,
        "val_spec":     float(h["best_spec"]) * PCT,
        "val_dice_nobg": float(h.get("best_dice_no_bg", np.nan)) * PCT,
        "best_epoch":   best_ep,
        "n_epochs":     n_epochs,
        "final_train_loss": float(train_loss[-1]) if len(train_loss) else np.nan,
        "n_test_patients":  len(h.get("test_patients", [])),
    })
    return parsed


val_rows = {}
for f in history_files:
    row = load_history(f)
    if row:
        val_rows[row["run"]] = row

print(f"Loaded {len(val_rows)} run histories from {len(history_files)} files")

if len(val_rows) != len(history_files):
    print()
    print(f'WARNING: {len(history_files) - len(val_rows)} file(s) produced no row.')
    print('         Distinct configurations may share a stored run_name, or a')
    print('         filename may be unparseable. See warnings above.')

if NAME_MISMATCH:
    print()
    print(f'NOTE: {len(NAME_MISMATCH)} file(s) store a run_name that differs from')
    print('      their filename. The FILENAME is authoritative here. This is')
    print('      expected where variants share a run_name (SpectralFormer ViT/CAF).')
    for fn, st in NAME_MISMATCH[:6]:
        print(f'        {fn}')
        print(f'          stored as: {st}')
    if len(NAME_MISMATCH) > 6:
        print(f'        ... and {len(NAME_MISMATCH) - 6} more')

if UNKNOWN_PREFIXES:
    print(f"\n⚠️  Unrecognised model prefixes (add to MODEL_REGISTRY): "
          f"{sorted(UNKNOWN_PREFIXES)}")

Loaded 111 run histories from 111 files

NOTE: 15 file(s) store a run_name that differs from
      their filename. The FILENAME is authoritative here. This is
      expected where variants share a run_name (SpectralFormer ViT/CAF).
        spectralformer_caf_ce_bal_fold1_vpfabelo
          stored as: spectralformer_ce_bal_fold1_vpfabelo
        spectralformer_caf_ce_bal_fold2_vpfabelo
          stored as: spectralformer_ce_bal_fold2_vpfabelo
        spectralformer_caf_ce_bal_fold3_vpfabelo
          stored as: spectralformer_ce_bal_fold3_vpfabelo
        spectralformer_caf_ce_bal_fold4_vpfabelo
          stored as: spectralformer_ce_bal_fold4_vpfabelo
        spectralformer_caf_ce_bal_fold5_vpfabelo
          stored as: spectralformer_ce_bal_fold5_vpfabelo
        spectralformer_vit_ce_bal_fold1_vpfabelo
          stored as: spectralformer_ce_bal_fold1_vpfabelo
        ... and 9 more


## 5 · Test metrics from confusion matrices

Metrics are recomputed from the saved matrices rather than parsed from the
markdown reports, so the definitions are explicit and auditable here.

For each class: sensitivity = TP/(TP+FN), specificity = TN/(TN+FP),
F1 = 2TP/(2TP+FP+FN). Macro F1 averages classes equally; the no-BG variant
excludes background so the score is not inflated by an easy majority class.

In [125]:
def metrics_from_cm(cm):
    """Per-class and macro metrics from a confusion matrix (rows=true)."""
    cm = np.asarray(cm, dtype=np.float64)
    n  = cm.sum()
    out, f1s = {}, []
    for i, lbl in enumerate(CM_LABELS):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = n - tp - fn - fp
        sens = tp / (tp + fn) if (tp + fn) else np.nan
        spec = tn / (tn + fp) if (tn + fp) else np.nan
        f1   = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else np.nan
        f1s.append(f1)
        out[f"{CLASS_ABBR[lbl]}_f1"]   = f1   * PCT
        out[f"{CLASS_ABBR[lbl]}_sens"] = sens * PCT
        out[f"{CLASS_ABBR[lbl]}_spec"] = spec * PCT
    nobg = [f for i, f in enumerate(f1s) if i != BG_INDEX]
    out["test_f1_all"]  = float(np.nanmean(f1s))  * PCT
    out["test_f1_nobg"] = float(np.nanmean(nobg)) * PCT
    out["test_oa"]      = float(np.trace(cm) / n) * PCT if n else np.nan
    out["test_tt_sens"] = out["TT_sens"]
    out["test_tt_f1"]   = out["TT_f1"]
    return out


_FOLD_KEY_RE = re.compile(r"fold[_\-]?(\d+)", re.IGNORECASE)


def load_test_eval(path):
    """Return {fold: metrics} from a test_eval archive, layout-tolerant."""
    z = np.load(path, allow_pickle=True)
    per_fold = {}

    for key in z.files:
        arr = np.asarray(z[key])
        m = _FOLD_KEY_RE.search(key)
        # Per-fold 2D confusion matrices
        if m and arr.ndim == 2 and arr.shape == (len(CM_LABELS), len(CM_LABELS)):
            per_fold[int(m.group(1))] = metrics_from_cm(arr)
        # Stacked (n_folds, C, C)
        elif arr.ndim == 3 and arr.shape[1:] == (len(CM_LABELS), len(CM_LABELS)):
            for i in range(arr.shape[0]):
                per_fold.setdefault(FOLDS[i] if i < len(FOLDS) else i + 1,
                                    metrics_from_cm(arr[i]))

    if not per_fold:
        warnings.warn(f"No confusion matrices recognised in {path.name} "
                      f"(keys: {list(z.files)})")
    return per_fold


test_rows = {}   # config prefix -> {fold: metrics}
for f in testeval_files:
    cfg = f.stem.replace("test_eval_", "")
    per_fold = load_test_eval(f)
    if per_fold:
        test_rows[cfg] = per_fold

print(f"Loaded test evaluations for {len(test_rows)} configuration(s):")
for cfg, folds in sorted(test_rows.items()):
    print(f"  {cfg:42} folds {sorted(folds)}")

Loaded test evaluations for 22 configuration(s):
  1dcnn_ce_bal_vpfabelo                      folds [1, 2, 3, 4, 5]
  1dcnn_ce_nobal_vpfabelo                    folds [1, 2, 3, 4, 5]
  1dcnn_ufl_bal_vpfabelo                     folds [1, 2, 3, 4, 5]
  1dcnn_ufl_nobal_vpfabelo                   folds [1, 2, 3, 4, 5]
  1dnn_ce_bal_vpfabelo                       folds [1, 2, 3, 4, 5]
  1dnn_ufl_bal_vpfabelo                      folds [1, 2, 3, 4, 5]
  1dnnfabelo_ce_bal_vpfabelo                 folds [1, 2, 3, 4, 5]
  1dnnfabelo_ce_nobal_vpfabelo               folds [1, 2, 3, 4, 5]
  1dnnfabelo_dl_bal_vpfabelo                 folds [1, 2, 3, 4, 5]
  1dnnfabelo_dl_nobal_vpfabelo               folds [1, 2, 3, 4, 5]
  1dnnfabelo_fl_bal_vpfabelo                 folds [1, 2, 3, 4, 5]
  1dnnfabelo_fl_nobal_vpfabelo               folds [1, 2, 3, 4, 5]
  1dnnfabelo_ufl_bal_vpfabelo                folds [1, 2, 3, 4, 5]
  1dnnfabelo_ufl_nobal_vpfabelo              folds [1, 2, 3, 4, 5]
  2dcnn_ce_ba

## 6 · Latency (optional)

Latency is a property of the architecture, not the loss function, so a single
measurement per model applies to all of its loss variants.

In [126]:
latency_by_model = {}
if LATENCY_JSON.exists():
    payload = json.loads(LATENCY_JSON.read_text())
    for r in payload.get("results", []):
        # Match the measured label back to a registry display name
        for _pfx, (_name, _t, _p, _ty) in MODEL_REGISTRY.items():
            if _name.lower().replace("-", "") in r["label"].lower().replace("-", "").replace(" ", ""):
                latency_by_model[_name] = r["median_s"]
                break
    print(f"Latency on {payload.get('device', 'unknown device')}:")
    for k, v in sorted(latency_by_model.items(), key=lambda kv: kv[1]):
        print(f"  {k:22} {v:9.3f} s")
else:
    print("No latency file — latency columns will show '--'.")

No latency file — latency columns will show '--'.


## 7 · Per-fold results table

One row per configuration per fold, keyed by run name. `Test` columns are
blank where that configuration has not been test-evaluated.

In [127]:
def fmt(v, dp=1):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "--"
    return f"{v:.{dp}f}"


records = []
for run, row in val_rows.items():
    rec = dict(row)
    tv = test_rows.get(row["config"], {}).get(row["fold"])
    if tv:
        rec.update(tv)
        rec["has_test"] = True
    else:
        rec["has_test"] = False
    rec["latency_s"] = latency_by_model.get(row["model"])
    records.append(rec)

# Sort by tier, then model, loss, fold
records.sort(key=lambda r: (r["tier"], r["model"], r["loss"], r["fold"]))

t = PrettyTable()
t.field_names = [
    "Run (ID)", "Model", "Tier", "Loss", "Fold",
    "Val F1-noBG", "Val Sens", "Ep", "N-ep",
    "Test F1-noBG", "Test OA", "Test TT sens", "Test TT F1",
]
t.align = "r"
t.align["Run (ID)"] = "l"
t.align["Model"] = "l"

for r in records:
    t.add_row([
        r["run"], r["model"], r["tier"], r["loss"], r["fold"],
        fmt(r["val_f1_nobg"]), fmt(r["val_sens"]),
        r["best_epoch"], r["n_epochs"],
        fmt(r.get("test_f1_nobg")), fmt(r.get("test_oa")),
        fmt(r.get("test_tt_sens")), fmt(r.get("test_tt_f1")),
    ])

print(f"{len(records)} run(s) · all metrics in %")
print(t)

111 run(s) · all metrics in %
+-------------------------------------------+--------------------+------+------+------+-------------+----------+----+------+--------------+---------+--------------+------------+
| Run (ID)                                  | Model              | Tier | Loss | Fold | Val F1-noBG | Val Sens | Ep | N-ep | Test F1-noBG | Test OA | Test TT sens | Test TT F1 |
+-------------------------------------------+--------------------+------+------+------+-------------+----------+----+------+--------------+---------+--------------+------------+
| 1dnn_ce_bal_fold1_vpfabelo                | 1D-NN-Baseline     |    1 |   CE |    1 |        60.8 |     79.4 |  6 |   21 |         65.9 |    71.1 |         85.7 |       34.3 |
| 1dnn_ce_bal_fold2_vpfabelo                | 1D-NN-Baseline     |    1 |   CE |    2 |        86.7 |     90.3 | 30 |   45 |         72.3 |    81.8 |         52.0 |       44.6 |
| 1dnn_ce_bal_fold3_vpfabelo                | 1D-NN-Baseline     |    1 |   CE |

## 8 · Per-configuration aggregate

Median ± population standard deviation across folds, matching the aggregation
used in the training logs and test-evaluation reports.

In [128]:
def agg(vals):
    vals = [v for v in vals if v is not None and not np.isnan(v)]
    if not vals:
        return None, None
    return float(np.median(vals)), float(np.std(vals))


by_config = OrderedDict()
for r in records:
    by_config.setdefault(r["config"], []).append(r)

a = PrettyTable()
a.field_names = [
    "Configuration (ID)", "Model", "Tier", "Loss", "Params", "Folds",
    "Val F1-noBG", "Test F1-noBG", "Test TT sens", "Test TT F1",
    "Latency (s)",
]
a.align = "r"
a.align["Configuration (ID)"] = "l"
a.align["Model"] = "l"

agg_records = []
for cfg, rows in sorted(by_config.items(),
                        key=lambda kv: (kv[1][0]["tier"], kv[1][0]["model"],
                                        kv[1][0]["loss"])):
    base = rows[0]
    vf1_m, vf1_s = agg([r["val_f1_nobg"] for r in rows])
    tf1_m, tf1_s = agg([r.get("test_f1_nobg") for r in rows])
    tts_m, tts_s = agg([r.get("test_tt_sens") for r in rows])
    ttf_m, ttf_s = agg([r.get("test_tt_f1") for r in rows])

    def pm(m, s):
        return "--" if m is None else f"{m:.1f} ± {s:.1f}"

    params = f"{base['params']:,}" if base["params"] else "--"
    lat = base["latency_s"]
    agg_records.append({
        "config": cfg, "model": base["model"], "tier": base["tier"],
        "loss": base["loss"], "params": base["params"], "n_folds": len(rows),
        "val_f1_nobg": vf1_m, "val_f1_nobg_std": vf1_s,
        "test_f1_nobg": tf1_m, "test_f1_nobg_std": tf1_s,
        "test_tt_sens": tts_m, "test_tt_sens_std": tts_s,
        "test_tt_f1": ttf_m, "test_tt_f1_std": ttf_s,
        "latency_s": lat,
    })
    a.add_row([cfg, base["model"], base["tier"], base["loss"], params,
               len(rows), pm(vf1_m, vf1_s), pm(tf1_m, tf1_s),
               pm(tts_m, tts_s), pm(ttf_m, ttf_s),
               "--" if lat is None else f"{lat:.3f}"])

print(a)

+-------------------------------------+--------------------+------+------+------------+-------+-------------+--------------+--------------+-------------+-------------+
| Configuration (ID)                  | Model              | Tier | Loss |     Params | Folds | Val F1-noBG | Test F1-noBG | Test TT sens |  Test TT F1 | Latency (s) |
+-------------------------------------+--------------------+------+------+------------+-------+-------------+--------------+--------------+-------------+-------------+
| 1dnn_ce_bal_vpfabelo                | 1D-NN-Baseline     |    1 |   CE | 17,000,000 |     5 |  67.6 ± 9.1 |   68.5 ± 2.5 |  51.4 ± 15.4 |  41.9 ± 5.9 |          -- |
| 1dnn_ufl_bal_vpfabelo               | 1D-NN-Baseline     |    1 |  UFL | 17,000,000 |     5 |  67.6 ± 8.2 |   69.5 ± 1.6 |  52.6 ± 10.2 |  42.7 ± 3.9 |          -- |
| 1dnnfabelo_ce_bal_vpfabelo          | 1D-NN-Fabelo       |    1 |   CE |      4,936 |     5 |  70.5 ± 6.3 |   70.8 ± 3.2 |   62.4 ± 9.9 |  44.6 ± 7.1 |       

## 9 · Coverage against the planned configuration space

Reports which planned configurations are trained, test-evaluated, both, or
neither. Untrained tiers (3D, transformer) appear here explicitly rather than
being silently absent.

In [129]:
PLANNED_MODELS = [
    ("1D-NN-Fabelo",       1), ("1D-NN-Baseline",     1),
    ("1D-CNN-Hu",          2),
    ("2D-CNN-Fabelo",      3), ("2D-CNN-Simple",      3),
    ("2D-CNN-LeeEtAl",     3),
    ("3D-CNN-Hamida",      4), ("HybridSN",           4),
    ("SpectralFormer-ViT", 5), ("SpectralFormer-CAF", 5),
]
PLANNED_LOSSES = ["CE", "FL", "DL", "UFL"]

trained  = {(r["model"], r["loss"]) for r in records}
tested   = {(r["model"], r["loss"]) for r in records if r["has_test"]}

c = PrettyTable()
c.field_names = ["Model", "Tier"] + PLANNED_LOSSES
c.align = "l"

STATUS = {"both": "train+test", "train": "train only", "none": "--"}
missing_trained, missing_tested = [], []

for model, tier in PLANNED_MODELS:
    cells_row = []
    for loss in PLANNED_LOSSES:
        if (model, loss) in tested:
            cells_row.append(STATUS["both"])
        elif (model, loss) in trained:
            cells_row.append(STATUS["train"])
            missing_tested.append(f"{model} x {loss}")
        else:
            cells_row.append(STATUS["none"])
    c.add_row([model, tier] + cells_row)

print(c)

n_planned = len(PLANNED_MODELS) * len(PLANNED_LOSSES)
print(f"\nPlanned configurations : {n_planned}")
print(f"Trained                : {len(trained)}")
print(f"Test-evaluated         : {len(tested)}")

untrained_tiers = sorted({t for m, t in PLANNED_MODELS
                          if not any(mm == m for mm, _ in trained)})
if untrained_tiers:
    print(f"\n⚠️  Models with no training runs found "
          f"(tiers {untrained_tiers}):")
    for m, t in PLANNED_MODELS:
        if not any(mm == m for mm, _ in trained):
            print(f"     - {m} (tier {t})")
if missing_tested:
    print(f"\n⚠️  Trained but not test-evaluated:")
    for x in sorted(set(missing_tested)):
        print(f"     - {x}")

+--------------------+------+------------+------------+------------+------------+
| Model              | Tier | CE         | FL         | DL         | UFL        |
+--------------------+------+------------+------------+------------+------------+
| 1D-NN-Fabelo       | 1    | train+test | train+test | train+test | train+test |
| 1D-NN-Baseline     | 1    | train+test | --         | --         | train+test |
| 1D-CNN-Hu          | 2    | train+test | --         | --         | train+test |
| 2D-CNN-Fabelo      | 3    | train+test | --         | --         | train+test |
| 2D-CNN-Simple      | 3    | train+test | --         | --         | --         |
| 2D-CNN-LeeEtAl     | 3    | train+test | --         | --         | train+test |
| 3D-CNN-Hamida      | 4    | --         | --         | --         | --         |
| HybridSN           | 4    | --         | --         | --         | --         |
| SpectralFormer-ViT | 5    | train+test | --         | --         | train+test |
| SpectralFormer

## 10 · Export

In [130]:
import csv

# Per-fold markdown
per_fold_md = OUT_DIR / "summary_per_fold.md"
with open(per_fold_md, "w") as f:
    f.write("# Per-fold results — all runs\n\n")
    f.write("All metrics in %. `--` = not evaluated.\n\n")
    f.write("| Run (ID) | Model | Tier | Loss | Fold | Val F1-noBG | Val Sens "
            "| Best ep | Epochs | Test F1-noBG | Test OA | Test TT sens | Test TT F1 |\n")
    f.write("|" + "---|" * 13 + "\n")
    for r in records:
        f.write(f"| `{r['run']}` | {r['model']} | {r['tier']} | {r['loss']} | "
                f"{r['fold']} | {fmt(r['val_f1_nobg'])} | {fmt(r['val_sens'])} | "
                f"{r['best_epoch']} | {r['n_epochs']} | "
                f"{fmt(r.get('test_f1_nobg'))} | {fmt(r.get('test_oa'))} | "
                f"{fmt(r.get('test_tt_sens'))} | {fmt(r.get('test_tt_f1'))} |\n")

# Per-config markdown
per_cfg_md = OUT_DIR / "summary_per_config.md"
with open(per_cfg_md, "w") as f:
    f.write("# Per-configuration aggregate\n\n")
    f.write("Median ± population std across folds. All metrics in %.\n\n")
    f.write("| Configuration (ID) | Model | Tier | Loss | Params | Folds | "
            "Val F1-noBG | Test F1-noBG | Test TT sens | Test TT F1 | Latency (s) |\n")
    f.write("|" + "---|" * 11 + "\n")
    for r in agg_records:
        def pm(m, s):
            return "--" if m is None else f"{m:.1f} ± {s:.1f}"
        params = f"{r['params']:,}" if r["params"] else "--"
        lat = "--" if r["latency_s"] is None else f"{r['latency_s']:.3f}"
        f.write(f"| `{r['config']}` | {r['model']} | {r['tier']} | {r['loss']} | "
                f"{params} | {r['n_folds']} | "
                f"{pm(r['val_f1_nobg'], r['val_f1_nobg_std'])} | "
                f"{pm(r['test_f1_nobg'], r['test_f1_nobg_std'])} | "
                f"{pm(r['test_tt_sens'], r['test_tt_sens_std'])} | "
                f"{pm(r['test_tt_f1'], r['test_tt_f1_std'])} | {lat} |\n")

# CSVs
per_fold_csv = OUT_DIR / "summary_per_fold.csv"
fields = ["run", "config", "model", "tier", "loss", "balance", "fold", "params",
          "val_f1_nobg", "val_f1_all", "val_sens", "val_spec",
          "best_epoch", "n_epochs",
          "test_f1_nobg", "test_f1_all", "test_oa",
          "test_tt_sens", "test_tt_f1", "latency_s"]
with open(per_fold_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
    w.writeheader()
    w.writerows(records)

per_cfg_csv = OUT_DIR / "summary_per_config.csv"
with open(per_cfg_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(agg_records[0].keys()))
    w.writeheader()
    w.writerows(agg_records)

for p in [per_fold_md, per_cfg_md, per_fold_csv, per_cfg_csv]:
    print(f"  written -> {p}")

  written -> ../results/summary_per_fold.md
  written -> ../results/summary_per_config.md
  written -> ../results/summary_per_fold.csv
  written -> ../results/summary_per_config.csv
